# 04. Extended hypothesis tests -- Kruskal-Wallis + Mann-Whitney U (Appendix S.3)

Pools off-diagonal ARI values from per-asset cross-frequency matrices
into three frequency-pair categories (adjacent intraday, non-adjacent
intraday, intraday-daily) and tests whether the categories differ.

This is a *post-pipeline* test: the inputs are the cached per-asset
`{ASSET}_cross_freq_ari.csv` files. The notebook walks through the
pooling logic in-process so the reader can audit which (fa, fb) pairs
land in which category.

**Re-run command**: `python run.py extended_hypothesis_tests`

In [1]:
from __future__ import annotations
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

PROJECT = Path.cwd()
_nb_file = globals().get("__vsc_ipynb_file__")
if _nb_file is not None:
    PROJECT = Path(_nb_file).resolve().parent.parent
elif PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
sys.path.insert(0, str(PROJECT))

OUT = PROJECT / "outputs"
OUT_2022 = PROJECT / "outputs_2022"
DATA = PROJECT / "data"
DATA_2022 = PROJECT / "data_2022"

pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 50)

## Step 1 -- show the frequency-pair classification

`src.experiments.exp_03_hypothesis_tests._pair_type` classifies each
unordered pair on the ladder 5m < 15m < 1h < 1d.

In [2]:
from src.experiments.exp_03_hypothesis_tests import _pair_type
from src.core.config import FREQS

rows = []
for i, fa in enumerate(FREQS):
    for j, fb in enumerate(FREQS):
        if j <= i:
            continue
        rows.append({"fa": fa, "fb": fb, "pair_type": _pair_type(fa, fb)})
pd.DataFrame(rows)

,fa,fb,pair_type
0,5m,15m,adjacent_intraday
1,5m,1h,nonadjacent_intraday
2,5m,1d,intraday_daily
3,15m,1h,nonadjacent_intraday
4,15m,1d,intraday_daily
5,1h,1d,intraday_daily


## Step 2 -- pool off-diagonal ARI from cached per-asset matrices

In [3]:
from src.experiments.exp_03_hypothesis_tests import pooled_ari_groups

out_dirs = [OUT]
if (OUT_2022 / f"{ 'SPY' }_cross_freq_ari.csv").exists():
    out_dirs.append(OUT_2022)

groups = pooled_ari_groups(out_dirs, ['SPY', 'USDJPY', 'CL', 'GLD'])
pd.DataFrame([
    {"group": k, "n": len(v), "mean": float(np.mean(v)) if v else float("nan")}
    for k, v in groups.items()
])

,group,n,mean
0,adjacent_intraday,7,0.595
1,nonadjacent_intraday,14,0.175
2,intraday_daily,21,-0.011


## Step 3 -- run Kruskal-Wallis + one-sided Mann-Whitney U

The two MWU tests use `alternative="greater"` because the pre-registered
direction (cross-frequency dissonance hypothesis) is that ARI decreases
monotonically as the resolution gap widens.

In [4]:
from scipy import stats

g1 = np.asarray(groups["adjacent_intraday"], dtype=float)
g2 = np.asarray(groups["nonadjacent_intraday"], dtype=float)
g3 = np.asarray(groups["intraday_daily"], dtype=float)

kw_stat, kw_p = stats.kruskal(g1, g2, g3)
u12, p12 = stats.mannwhitneyu(g1, g2, alternative="greater")
u23, p23 = stats.mannwhitneyu(g2, g3, alternative="greater")

pd.DataFrame([{
    "kw_H": kw_stat, "kw_p": kw_p,
    "MWU_adj_vs_nonadj_U": u12, "MWU_adj_vs_nonadj_p": p12,
    "MWU_nonadj_vs_daily_U": u23, "MWU_nonadj_vs_daily_p": p23,
}]).round(4)

,kw_H,kw_p,MWU_adj_vs_nonadj_U,MWU_adj_vs_nonadj_p,MWU_nonadj_vs_daily_U,MWU_nonadj_vs_daily_p
0,32.597,0.0,98.0,0.0,287.0,0.0


## Cached full-scale result -- `outputs/hypothesis_tests.csv`

In [5]:
p = OUT / "hypothesis_tests.csv"
display(pd.read_csv(p).round(4) if p.exists() else Markdown(f"`{p}` missing"))

,mean_adjacent_intraday,n_adjacent_intraday,mean_nonadjacent_intraday,n_nonadjacent_intraday,mean_intraday_daily,n_intraday_daily,kruskal_H,kruskal_p,mannwhitney_adj_vs_nonadj_U,mannwhitney_adj_vs_nonadj_p,mannwhitney_adj_vs_nonadj_q_bh,mannwhitney_nonadj_vs_daily_U,mannwhitney_nonadj_vs_daily_p,mannwhitney_nonadj_vs_daily_q_bh
0,0.595,7,0.175,14,-0.011,21,32.597,0.0,98.0,0.0,0.0,287.0,0.0,0.0
